In [1]:
# !pip install langchain-community langchain-core langchain-openai langchain-classic
# !pip install -U langchain-chroma
# !pip install sentence-transformers chromadb
# !pip install ragas deepeval
# !pip install rank_bm25
# !pip install -q 'transformers>=4.51.3,<5.17.0'

In [ ]:
import os
import re
import json
import uuid
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from langchain_core.load import dumps, loads
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""

# 1) Chunking

Chunking is one of the biggest levers for RAG quality.
The core question: "Do we need chunking at all for this corpus?"

Decision framework:
  - Short, single-purpose docs (FAQs, tickets): No chunking needed
  - Long, multi-topic docs (manuals, reports): Chunking is essential

## 1.1 Fixed Size Chunking

The simplest approach: split every N characters with overlap.

- Pro: Predictable chunk sizes, easy to reason about.
- Con: Splits mid-sentence, breaks semantic coherence.

In [3]:
fixed_splitter = CharacterTextSplitter(
    chunk_size=500,  # num_samples characters per chunk
    chunk_overlap=50,  # overlap to preserve boundary context
    separator="\n",  # try to split on newlines first
)

## 1.2 Recursive Chunking

* Tries to split on paragraph boundaries first, then sentences, then words.
* Preserves semantic units better than fixed-size.

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    # Priority: paragraph → newline → sentence → word → character
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Example usage:
# Each split retains .metadata from the parent document
# splits = recursive_splitter.split_documents(documents)

## 1.3 Structure-Aware Chunking
* For markdown/HTML documents, split along structural boundaries.
* Preserves section context and adds header info as metadata.

In [5]:
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# For oversized sections, chain with recursive splitter:
# md_chunks = markdown_splitter.split_text(markdown_text)
# final_chunks = recursive_splitter.split_documents(md_chunks)

## 1.4 Semantic Chunking

* Group consecutive sentences whose embeddings are similar.
* Uses a similarity threshold to detect topic boundaries.

In [ ]:
def semantic_chunk(
    text: str,
    model: SentenceTransformer,
    threshold: float = 0.75,
    min_chunk_size: int = 100,
) -> list[str]:
    """
    Split text into semantically coherent chunks.

    Algorithm:
      1. Split text into sentences
      2. Embed each sentence
         # num_samples sentences → (num_samples, embed_dim)
      3. Compute cosine similarity between consecutive sentence embeddings
      4. Split where similarity drops below threshold

    Args:
        text: Full document text
        model: SentenceTransformer model for encoding
        threshold: Cosine similarity cutoff (lower = larger chunks)
        min_chunk_size: Minimum characters per chunk (merge small chunks)

    Returns:
        List of semantically coherent text chunks
    """
    # Split into sentences (simple regex; use spaCy/nltk for production)
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if len(sentences) <= 1:
        return [text]

    # Encode all sentences
    # Output: (num_samples, embed_dim) numpy array
    embeddings = model.encode(sentences, normalize_embeddings=True)

    # Compute cosine similarity between consecutive sentences
    # (num_samples-1,) similarity scores
    similarities = np.array(
        [np.dot(embeddings[i], embeddings[i + 1]) for i in range(len(embeddings) - 1)]
    )

    # Find split points where similarity drops below threshold
    chunks = []
    current_chunk = [sentences[0]]

    for i, sim in enumerate(similarities):
        if sim < threshold and len(" ".join(current_chunk)) >= min_chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i + 1]]
        else:
            current_chunk.append(sentences[i + 1])

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

## 1.5 Contextual Retrieval

* Problem: Individual chunks lose context when retrieved in isolation.
* Solution: Prepend a short document-level context to each chunk before embedding.

Steps:
  1. Chunk document normally
  2. For each chunk, use an LLM to generate a brief context blurb
     that situates the chunk within the full document
  3. Prepend the context to the chunk text before embedding

This makes each chunk self-contained and improves retrieval precision.

Then embed enriched_chunk instead of raw chunk_text

In [7]:
CONTEXTUAL_PROMPT = """
You are given a document and a specific chunk from that document.
Generate a short (2-3 sentence) context that explains where this chunk
fits within the overall document. Include the document title, section,
and any key entities referenced.

<document>
{document}
</document>

<chunk>
{chunk}
</chunk>

Provide ONLY the brief context, nothing else.
"""

# context = llm.invoke(CONTEXTUAL_PROMPT.format(document=full_doc, chunk=chunk))
# enriched_chunk = f"{context}\n\n{chunk}"

## 1.6 Late Chunking

* Problem: Standard chunking loses cross-sentence references (pronouns, etc.)
* Solution: Pass full document through a long-context embedding model first,
          then split the token-level embeddings into chunks and mean-pool.

* Pseudocode:
  1. full_embeddings = model.encode_tokens(full_document)  # (total_tokens, embed_dim)
  2. For each chunk boundary [start:end]:
       chunk_embedding = mean(full_embeddings[start:end])  # (embed_dim,)
  3. Store chunk_embedding in vector DB

Tradeoff: Higher efficiency than contextual retrieval (no LLM calls),
          but slightly lower semantic coherence than contextual retrieval.

# 2) EMBEDDING MODELS

## 2.1 Huggingface OpenSource Embedding

In [ ]:
# Lightweight model for prototyping

model_mini = SentenceTransformer("all-MiniLM-L6-v2")
texts = ["How does photosynthesis work?", "Plants convert sunlight into energy"]

# Encode texts into dense vectors
embeddings = model_mini.encode(texts, normalize_embeddings=True)
similarity = np.dot(embeddings[0], embeddings[1])
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.6456


## 2.3 OpenAI Embeddings with Matryoshka Dimensionality

Matryoshka embeddings: truncate to lower dimensions with minimal quality loss.

Full 3072-dim → 256-dim: only ~8% quality drop, 12x storage savings.

In [9]:
from langchain_openai import OpenAIEmbeddings

# Full-dimension embeddings
# 3072-dim
embed_full = OpenAIEmbeddings(model="text-embedding-3-large")

# Truncated embeddings (Matryoshka) — same model, lower storage
embed_small = OpenAIEmbeddings(
    model="text-embedding-3-large",
    # Truncate to first 256 dims
    dimensions=256,
)

# === Usage ===
vectors = embed_full.embed_documents(["Hello world", "Goodbye world"])
query_vec = embed_full.embed_query("Hi there")

# 3) VECTOR STORES & INDEXING

Vector databases store embeddings and enable fast approximate nearest neighbor
(ANN) search.

Key algorithm: HNSW (Hierarchical Navigable Small World) —
builds a multi-layer graph where search complexity is O(log N).

Production recommendations:
  - Prototyping: Chroma (in-process, zero config)
  - Production (<100M vectors): Qdrant or Weaviate
  - Massive scale (billions): Milvus / Zilliz Cloud
  - Already using PostgreSQL: pgvector (good for <50M vectors)

## 3.1 Chroma

In [ ]:
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create sample documents with metadata
docs = [
    Document(
        page_content="Neural networks learn hierarchical representations of data.",
        metadata={"source": "ml_textbook", "chapter": 3, "topic": "deep_learning"},
    ),
    Document(
        page_content="Gradient descent optimizes model parameters by following the loss gradient.",
        metadata={"source": "ml_textbook", "chapter": 2, "topic": "optimization"},
    ),
    Document(
        page_content="Transformers use self-attention to process sequences in parallel.",
        metadata={"source": "ml_textbook", "chapter": 5, "topic": "transformers"},
    ),
]

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="ml_textbook",
    persist_directory="./chroma_db",
)
print(f"Vector DB contains {vectordb._collection.count()} documents")

Vector DB contains 18 documents


In [11]:
# Different search strategies for different use cases.

# Basic similarity search (cosine distance)
retriever_basic = vectordb.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)

# MMR (Maximal Marginal Relevance): Balances relevance with diversity
# Reduces redundancy in retrieved documents
retriever_mmr = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,  # Return 3 documents
        "fetch_k": 10,  # Fetch 10 candidates first
        "lambda_mult": 0.7,  # 0=max diversity, 1=max relevance
    },
)

# Similarity with score threshold: Only return high-confidence matches
retriever_threshold = vectordb.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 5,
        "score_threshold": 0.5,  # Minimum similarity score
    },
)

# Metadata filtering: Restrict search to specific document subsets
retriever_filtered = vectordb.as_retriever(
    search_kwargs={
        "k": 3,
        # exact match
        "filter": {"topic": "transformers"},
        # # OR compound filters:
        # "filter": {"$and": [
        #     {"chapter": {"$gte": 3}},
        #     {"source": {"$eq": "ml_textbook"}}
        # ]}
    }
)
print(retriever_filtered)

tags=['Chroma', 'HuggingFaceEmbeddings'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x14551cc50> search_kwargs={'k': 3, 'filter': {'topic': 'transformers'}}


# 4) RETRIEVAL METHODS

Retrieval quality is the #1 factor in RAG performance.

"Garbage in, garbage out" — if you retrieve irrelevant chunks, no LLM can save you.

## 4.1 Basic Vector Similarity

In [12]:
query = "How do transformers process data?"
results = vectordb.similarity_search_with_score(query, k=3)
for doc, score in results:
    print(f"Score: {score:.4f} | {doc.page_content[:80]}...")

Score: 0.7880 | Transformers use self-attention to process sequences in parallel....
Score: 0.7880 | Transformers use self-attention to process sequences in parallel....
Score: 0.7880 | Transformers use self-attention to process sequences in parallel....


## 4.2 Hybrid Search (BM25 + Dense)

Combines keyword-based (BM25) and semantic (dense vector) search.

Why:
- Dense search captures meaning ("car" ≈ "automobile") but misses exact terms.
- BM25 captures exact keywords (".concat()" method) but misses synonyms.

BM25 Formula:
$$
\text{score}(D, Q) = \sum_{i=1}^{|Q|} \text{IDF}(q_i) \cdot \frac{\text{TF}(q_i, D) \cdot (k_1 + 1)}{\text{TF}(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

Where:
  - TF(qi, D): Term frequency of query term qi in document D
  - IDF(qi): Inverse document frequency = log((N - DF + 0.5) / (DF + 0.5) + 1)
  - |D|: Document length, avgdl: Average document length
  - k1 ≈ 1.2: Controls TF saturation (higher = more weight to repeated terms)
  - b ≈ 0.75: Controls document length normalization (0=no penalty, 1=full penalty)

Key insight from BM25:
  TF/(TF + k) provides diminishing returns — it's always better to have one instance
  of each query term than two instances of just one of them.

In [13]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# BM25 retriever (keyword-based, operates on raw text)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3

# Dense vector retriever
dense_retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3})

# Ensemble retriever: Combine BM25 + dense with equal weights
# Uses Reciprocal Rank Fusion internally to merge result lists
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    # Slightly favor semantic search
    weights=[0.4, 0.6],
)

hybrid_results = hybrid_retriever.invoke("gradient descent optimization")

## 4.3 Multi-Vector Retrieval

Create multiple vector representations per document to improve recall.

Patterns:
  - Parent Document: Embed small chunks, retrieve parent (large) chunks
  - Summary: Embed document summaries, retrieve full documents
  - Hypothetical Questions: Embed LLM-generated questions, retrieve source docs
  - HyDE: Generate hypothetical answer, embed it, retrieve real documents

In [14]:
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_community.embeddings import HypotheticalDocumentEmbedder
from langchain_openai import ChatOpenAI

### 4.3.1 Parent Document Retrieval

Choosing the right chunk size is a balancing act between precision and context:

*   **Small Chunks (e.g., 128-256 tokens):**
    *   *Pros:* High retrieval precision. The embedding strictly represents a single specific idea, minimizing "noise" in the vector space.
    *   *Cons:* Lacks context. An LLM might receive a highly relevant sentence but lack the surrounding text needed to fully understand or synthesize the answer.
*   **Large Chunks (e.g., 512-1024+ tokens):**
    *   *Pros:* Provides rich context for the LLM to generate comprehensive answers. Preserves references and entity relationships.
    *   *Cons:* Lower retrieval precision. The embedding is an average of many ideas; specific facts might get "diluted" and missed during semantic search.

*Best Practice:* Use **Parent-Child Chunking** to get the best of both worlds—embed small chunks for precise retrieval, but pass the larger parent chunk to the LLM for context.

Embed small chunks (better precision) → return parent chunk (more context)

In [15]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

# id_key links child chunks to parent documents
id_key = "doc_id"


def build_parent_retriever(documents, vectorstore, embedding):
    """
    Build a parent-document retriever.

    Strategy:
      1. Split documents into large parent chunks
      2. Split each parent into smaller child chunks
      3. Embed and index child chunks (for precise matching)
      4. At query time: retrieve child → look up parent → return parent

    This gives you the precision of small-chunk embedding with the
    context of large-chunk retrieval.
    """
    # Split into parent chunks
    parent_docs = parent_splitter.split_documents(documents)
    doc_ids = [str(uuid.uuid4()) for _ in parent_docs]

    # Split each parent into child chunks, linking via doc_id
    child_docs = []
    for i, parent in enumerate(parent_docs):
        children = child_splitter.split_documents([parent])
        for child in children:
            child.metadata[id_key] = doc_ids[i]
        child_docs.extend(children)

    # Add child embeddings to vector store
    vectorstore.add_documents(child_docs)

    # Set up byte store for parent documents
    store = InMemoryByteStore()
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        byte_store=store,
        id_key=id_key,
    )

    # Store parent documents in byte store (keyed by doc_id)
    retriever.docstore.mset(list(zip(doc_ids, parent_docs)))

    return retriever

### 4.3.2 Summary-Based Multi-Vector Retrieval

- Embed document summaries (which distill key themes) instead of raw chunks.
- At query time: match query → summary embedding → return full source document.
- Why:
Summaries capture the "gist" more accurately than any single chunk,leading to better retrieval for broad or abstract queries.

---

1. retriever.invoke("What are the key economic trends?")
2. Embeds query → searches summary_vectorstore → finds matching summary
3. Reads doc_id from the matched summary's metadata
4. Looks up doc_id in byte store → returns the full source document

In [16]:
id_key = "doc_id"

# --- Step 1: Generate summaries ---
summary_chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Write a concise summary (3-5 sentences) of the following document. "
        "Focus on the key topics, entities, and conclusions.\n\n{doc}"
    )
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

summaries = summary_chain.batch(docs, {"max_concurrency": 4})

# --- Step 2: Assign unique IDs linking summaries to their source documents ---
doc_ids = [str(uuid.uuid4()) for _ in docs]

# --- Step 3: Create summary Documents tagged with doc_id ---
# These are what get embedded and searched against
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# --- Step 4: Build vector store over summaries ---
# num_docs summary strings → embedded as (num_docs, embed_dim)
summary_vectorstore = Chroma.from_documents(
    documents=summary_docs,
    embedding=embedding_model,
    collection_name="doc_summaries",
)

# --- Step 5: Build byte store and populate it with full documents ---
# The byte store is a simple key-value store: doc_id → full Document
store = InMemoryByteStore()

# --- Step 6: Wire them together into the MultiVectorRetriever ---
retriever = MultiVectorRetriever(
    vectorstore=summary_vectorstore,  # Searches here (summaries)
    byte_store=store,  # Looks up full docs here
    id_key=id_key,  # Links them via this metadata key
)

# --- Step 7: Populate the byte store with full source documents ---
# This is the critical step — without this, the retriever has nothing to return
# Maps: doc_ids[0] → source_documents[0], doc_ids[1] → source_documents[1], ...
retriever.docstore.mset(list(zip(doc_ids, docs)))

### 4.3.3 Hypothetical Questions Multi-Vector Retrieval

Pipeline:
- Document → LLM generates questions → Embed questions → Store in VectorDB
- Query → Match question → Lookup doc_id → Return Full Document

Why query-to-question matching works better than query-to-passage:

- User asks:     "What causes inflation?"

- Generated Q:   "What are the drivers of inflation?"     ← high similarity

- Raw passage:   "Rising prices occur when monetary..."   ← lower similarity

In [17]:
question_gen_chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Read the following document and generate exactly 3 diverse questions "
        "that this document could answer. Each question should approach the "
        "content from a different angle.\n\n"
        "Document:\n{doc}\n\n"
        'Output ONLY a JSON list: ["Q1?", "Q2?", "Q3?"]'
    )
    | ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    | StrOutputParser()
    | (lambda x: json.loads(x))
)

# Output: list of num_docs lists, each containing 3 question strings
# e.g., [["Q1a?", "Q1b?", "Q1c?"], ["Q2a?", "Q2b?", "Q2c?"], ...]
hypothetical_questions = question_gen_chain.batch(docs, {"max_concurrency": 4})
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Flatten questions into Documents, each tagged with parent doc_id ---
# If document_0 generated ["Q1?", "Q2?", "Q3?"], we create 3 separate Documents
# all sharing doc_ids[0], so any matched question leads back to document_0
question_docs = []
for i, q_list in enumerate(hypothetical_questions):
    for question in q_list:
        question_docs.append(
            Document(page_content=question, metadata={id_key: doc_ids[i]})
        )


# --- Build vector store over generated questions ---
# (num_docs * 3) question strings → embedded as (num_docs*3, embed_dim)
question_vectorstore = Chroma.from_documents(
    documents=question_docs,
    embedding=embedding_model,
    collection_name="hypothetical_questions",
)

## 4.4 HyDE (Hypothetical Document Embeddings)

- Problem: User queries are often short and vague ("inflation causes").
- Document passages are long and detailed ("Consumer prices rise when...").
- Insight: A hypothetical ANSWER to the query is linguistically closer to the actual documents than the query itself.
- The hypothetical answer doesn't need to be correct — it just needs to be in the same linguistic style as the real documents so that its embedding lands near the right neighborhood in vector space.

In [18]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "You are an expert researcher. Given the question below, write a short "
    "paragraph (3-5 sentences) that would be a good answer. "
    "Write in a factual, encyclopedic style as if it were from a textbook. "
    "Do NOT say 'I don't know'. Just write a plausible answer.\n\n"
    "Question: {question}\n\n"
    "Answer:"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
hyde_generator = HYDE_PROMPT | llm | StrOutputParser()


def hyde_retrieve(
    query: str, vectorstore, embedding_model, generator=hyde_generator, k: int = 5
) -> list:
    """
    Retrieve documents using Hypothetical Document Embeddings (HyDE).

    Pipeline:
      1. LLM generates a hypothetical answer to the query
      2. Embed the hypothetical answer (NOT the original query)
         # 1 hypothetical string → (1, embed_dim)
      3. Search vector store using the hypothetical embedding
         # (1, embed_dim) → top-k real Documents
      4. Return the real documents (NOT the hypothetical answer)

    Args:
        query: User's original question (possibly vague)
        vectorstore: Chroma/Qdrant/etc. with real document embeddings
        embedding_model: Same model used to build the vector store
        generator: LLM chain that produces hypothetical answers
        k: Number of documents to retrieve

    Returns:
        List of real Document objects from the vector store
    """

    # Generate hypothetical answer
    # Output: hypothetical passage string (may be factually wrong, that's OK)
    hypothetical_doc = generator.invoke({"question": query})

    # Embed the hypothetical answer
    # This embedding lands in the "document neighborhood" of vector space,
    # closer to real documents than the original short query would
    hyde_embedding = embedding_model.embed_query(hypothetical_doc)

    # Search the vector store using the hypothetical embedding
    # Output: top-k (Document, score) pairs
    results = vectorstore.similarity_search_by_vector(hyde_embedding, k=k)

    return results

In [19]:
# LangChain's built-in HyDE wrapper
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    # Built-in prompt template for general queries, Or provide a custom prompt:
    prompt_key="web_search",
)

# hyde_embeddings.embed_query("What causes inflation?")
# → First generates a hypothetical answer, then embeds it

Using a deprecated class. Please use `from langchain_classic.chains import HypotheticalDocumentEmbedder` instead


## 4.5 Query Rewriting

* Problem: User queries are often noisy, ambiguous, or poorly phrased.

* Solution: Use an LLM to rewrite the query before retrieval.

In [20]:
rewrite_prompt = ChatPromptTemplate.from_template(
    "Rewrite the following query to be more specific and searchable. "
    "Remove irrelevant context. Output ONLY the improved query.\n\n"
    "Original: {query}\nImproved:"
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
rewriter = rewrite_prompt | llm | StrOutputParser()
clean_query = rewriter.invoke(
    {"query": "man that trial was wild! What's Meta revenue Q3 2023?"}
)
print(clean_query)

Meta revenue Q3 2023


## 4.6 Multi-Query Fusion with Reciprocal Rank Fusion (RRF)

Generate multiple query variants, retrieve for each, merge results with RRF.

Reciprocal Rank Fusion formula:
  RRF_score(d) = Σ 1 / (k + rank_i(d))

Where:
  - k: Smoothing constant (typically 60)
  - rank_i(d): Rank of document d in result list i
  - Higher k = less emphasis on top-ranked documents

Why RRF works:
  - Documents appearing in multiple result lists get boosted
  - Robust to different scoring scales across retrievers
  - Simple, parameter-free (just k), and effective

In [21]:
def reciprocal_rank_fusion(results: list[list], k: int = 60) -> list[tuple]:
    """
    Merge multiple ranked document lists using Reciprocal Rank Fusion.

    Args:
        results: List of ranked document lists (one per query variant)
                 Each inner list: [Doc1, Doc2, ...] in descending relevance
        k: Smoothing constant (default 60). Higher k reduces the influence
           of rank position, making fusion more uniform.

    Returns:
        List of (document, fused_score) tuples, sorted by fused score descending

    Example:
        If document D appears at rank 1 in query_1 results and rank 3 in query_2:
        score(D) = 1/(60+1) + 1/(60+3) = 0.01639 + 0.01587 = 0.03226
    """
    fused_scores = {}

    for doc_list in results:
        for rank, doc in enumerate(doc_list):
            doc_str = dumps(doc)  # Serialize for deduplication
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0.0
            fused_scores[doc_str] += 1.0 / (rank + k)

    # Sort by fused score (descending) and deserialize
    reranked = [
        (loads(doc_str), score)
        for doc_str, score in sorted(
            fused_scores.items(), key=lambda x: x[1], reverse=True
        )
    ]
    return reranked


# Multi-query generation chain:
multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Generate 4 diverse search queries for the given question. "
            "Each query should approach the topic from a different angle.",
        ),
        ("user", "Question: {question}\nQueries (one per line):"),
    ]
)

generate_queries = (
    multi_query_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    | StrOutputParser()
    | (lambda x: x.strip().split("\n"))
)
queries = generate_queries.invoke({"question": "What causes inflation?"})
results = [retriever.invoke(q) for q in queries]
fused = reciprocal_rank_fusion(results)

## 4.7 Step-Back Prompting

For complex questions, first ask a broader "step-back" question.
Retrieve context for BOTH the original and step-back questions.

Example:
- Original: "Was ChatGPT around while Trump was president?"
- Step-back: "When was ChatGPT released and when was Trump president?"

The step-back question is easier to retrieve factual context for.

In [23]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

stepback_examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "What can the members of The Police do?",
    },
    {
        "input": "Jan Sindel was born in what country?",
        "output": "What is Jan Sindel's personal history?",
    },
]

stepback_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert at paraphrasing questions into broader, "
            "easier-to-answer step-back questions.",
        ),
        FewShotChatMessagePromptTemplate(
            example_prompt=ChatPromptTemplate.from_messages(
                [("human", "{input}"), ("ai", "{output}")]
            ),
            examples=stepback_examples,
        ),
        ("user", "{question}"),
    ]
)

step_back_chain = stepback_prompt | llm | StrOutputParser()
# Then retrieve for both original_question and step_back_question

## 4.8 Self-Query Retriever (Metadata-Aware)

Automatically extracts structured filters from natural language queries.

E.g., "sci-fi movies rated above 8.5 before 2000" → filter: genre="sci-fi" AND rating>8.5 AND year<2000

In [ ]:
from pydantic import BaseModel


class AttributeInfo(BaseModel):
    name: str
    description: str
    type: str


metadata_field_info = [
    AttributeInfo(name="topic", description="The topic of the content", type="string"),
    AttributeInfo(name="chapter", description="The chapter number", type="integer"),
    AttributeInfo(name="source", description="The source document", type="string"),
]

## 4.9 Self-RAG (Adaptive Retrieval)

<img src="https://selfrag.github.io/static/images/teaser_self_rag_v8.png" width=500>

Self-RAG trains the model to decide WHEN to retrieve and to CRITIQUE its outputs.
Uses special reflection tokens:
  - [Retrieve]: Should I retrieve? (yes/no)
  - [ISREL]: Is retrieved passage relevant? (relevant/irrelevant)
  - [ISSUP]: Is response supported by evidence? (fully/partially/no support)
  - [ISUSE]: Is response useful? (1-5 rating)

Flow:
  1. Given query, model decides if retrieval is needed
  2. If yes: retrieve → check relevance → generate with evidence
  3. If no: generate directly from parametric knowledge
  4. Self-critique: verify support and usefulness

Benefits: Reduces unnecessary retrieval, improves factuality.
Implementation: Requires fine-tuned model (e.g., Llama + special tokens).

## 4.10 Corrective RAG (CRAG)

<img src="https://miro.medium.com/v2/resize:fit:2000/1*qKV_BQ4X2cFVhU1DIMRtKw.png" width=500>

CRAG adds a verification step after retrieval:
  1. Retrieve documents
  2. Evaluate relevance of each document (LLM-as-judge or classifier)
  3. If relevant → use as context
     If ambiguous → refine query and web search as supplement
     If irrelevant → discard and fall back to web search
  4. Generate answer from verified context only

This prevents the model from being misled by irrelevant retrievals.

In [26]:
RELEVANCE_PROMPT = """
Given a user question and a retrieved document, evaluate if the document
contains information relevant to answering the question.

Question: {question}
Document: {document}

Output ONLY one of: "relevant", "ambiguous", "irrelevant"
"""


def crag_evaluate_documents(question, documents, llm):
    verified = []
    for doc in documents:
        judgment = (
            llm.invoke(
                RELEVANCE_PROMPT.format(question=question, document=doc.page_content)
            )
            .content.strip()
            .lower()
        )
        if judgment == "relevant":
            verified.append(doc)
        elif judgment == "ambiguous":
            # Optionally: refine query and web search
            pass
    if not verified:
        # Fallback: web search
        pass
    return verified

## 4.11 Agentic RAG

The most advanced pattern

An agent orchestrates the entire RAG pipeline:
  Plan → Route → Retrieve → Verify → Generate → Stop

Agent loop:
  1. PLAN: Decompose complex query into sub-questions
  2. ROUTE: For each sub-question, choose the right tool:
     - Vector DB search, graph query, SQL query, web search, calculator, etc.
  3. RETRIEVE: Execute the chosen tool
  4. VERIFY: Check relevance and coverage
  5. GENERATE: Synthesize answer from verified context
  6. STOP: Return answer with per-claim citations

Implementation: Use LangGraph or LlamaIndex Workflows for state management.

Simplified example with LangGraph:

In [27]:
from langgraph.graph import StateGraph, END
from typing import TypedDict


class RAGState(TypedDict):
    question: str
    sub_questions: list[str]
    documents: list
    verified_docs: list
    answer: str


def plan_node(state):
    # Decompose question into sub-questions
    ...


def retrieve_node(state):
    # Route each sub-question to appropriate retriever
    ...


def verify_node(state):
    # Check document relevance (CRAG-style)
    ...


def generate_node(state):
    # Generate answer with citations
    ...


graph = StateGraph(RAGState)
graph.add_node("plan", plan_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("verify", verify_node)
graph.add_node("generate", generate_node)

graph.add_edge("plan", "retrieve")
graph.add_edge("retrieve", "verify")
graph.add_edge("verify", "generate")
graph.add_edge("generate", END)

# 5) Reranking

Reranking is a two-stage retrieval pattern:

  - Stage 1: Fast retrieval (vector search) → top-K candidates (e.g., K=20)
  - Stage 2: Precise reranking (cross-encoder) → top-N results (e.g., N=5)

Why: Bi-encoders (embedding models) are fast but approximate.
     Cross-encoders see query AND document together, giving much better relevance
     scores — but are too slow to run on the full corpus.

Popular rerankers:
  - Cohere Rerank v3: API-based, strong performance
  - BGE-reranker-v2-m3: Open-source, multilingual
  - cross-encoder/ms-marco-MiniLM-L-6-v2: Lightweight, fast
  - Jina Reranker v2: Good balance of speed and quality
  - Qwen3-Reranker: Open-weight, multilingual

"Lost in the Middle" problem:
  LLMs tend to ignore information in the middle of long contexts.
  Solution: Reorder documents so most relevant are at the beginning and end.

## 5.1 Cross-Encoder Reranking

In [ ]:
class CrossEncoderReranker:
    """
    Rerank documents using a cross-encoder model.

    Unlike bi-encoders that encode query and document separately,
    cross-encoders process [query, document] pairs jointly through
    the full transformer, capturing fine-grained interactions.

    Input:  query string + list of document strings
    Output: documents sorted by relevance score (descending)
    """

    def __init__(self, model_name: str = "BAAI/bge-reranker-v2-m3"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()

    def rerank(
        self, query: str, documents: list[str], top_n: int = 5
    ) -> list[tuple[str, float]]:
        """
        Rerank documents by relevance to query.

        Args:
            query: User query string
            documents: List of document strings to rerank
            top_n: Number of top results to return

        Returns:
            List of (document, score) tuples, sorted by score descending

        Process:
            1. Create [query, document] pairs
               # num_samples pairs → tokenized (num_samples, seq_len)
            2. Forward pass through cross-encoder
               # Output: (num_samples, 1) logit scores
            3. Sort by score, return top_n
        """
        pairs = [[query, doc] for doc in documents]

        with torch.no_grad():
            # Tokenize all pairs
            # Input: list of num_samples [query, doc] pairs
            # Output: dict with input_ids (num_samples, seq_len), attention_mask, etc.
            inputs = self.tokenizer(
                pairs,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=512,
            )

            # Forward pass: cross-attention between query and document tokens
            # Input: (num_samples, seq_len) token IDs
            # Output: logits (num_samples,) — higher = more relevant
            scores = self.model(**inputs, return_dict=True).logits.view(-1).float()

        # Sort by score descending, return top_n
        scored_docs = list(zip(documents, scores.tolist()))
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        return scored_docs[:top_n]


# Usage:
reranker = CrossEncoderReranker()
results = reranker.rerank(
    query="What is gradient descent?",
    documents=["Gradient descent is...", "The sky is blue...", "SGD optimizes..."],
    top_n=2,
)
print(f'Result: {results}')

Result: [('Gradient descent is...', 6.39275598526001), ('SGD optimizes...', -8.559346199035645)]


## 5.2 Long Context Reorder
Mitigate "Lost in the Middle" by placing most relevant docs at edges.

In [29]:
def long_context_reorder(documents: list) -> list:
    """
    Reorder documents to combat "Lost in the Middle" effect.

    LLMs attend more to the beginning and end of their context.
    This reorders so that:
      - Most relevant documents are at the beginning and end
      - Least relevant documents are in the middle

    Input: documents sorted by relevance (index 0 = most relevant)
    Output: reordered documents [1st, 3rd, 5th, ..., 6th, 4th, 2nd]
    """
    if len(documents) <= 2:
        return documents

    reordered = []
    # Odd-indexed docs go to the front (less relevant)
    for i in range(1, len(documents), 2):
        reordered.append(documents[i])

    # Even-indexed docs go to the back (more relevant, reversed)
    for i in range(0, len(documents), 2):
        reordered.insert(0, documents[i]) if i == 0 else None

    # Simpler approach: interleave from edges
    result = []
    left, right = 0, len(documents) - 1
    while left <= right:
        result.append(documents[left])
        if left != right:
            result.append(documents[right])
        left += 1
        right -= 1
    return result

# 6) Generation

The final step: given retrieved context, generate an answer.

Key prompt engineering principles for RAG:
  1. Clearly separate context from question
  2. Instruct the model to only use provided context
  3. Tell it to say "I don't know" if context is insufficient
  4. Ask for citations/references to specific chunks
  5. Set a concise output length

In [30]:
from langchain_core.runnables import RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_template("""\
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say "I don't have enough
information to answer this question."

Cite your sources by referencing [Source N] where N is the source number.

Context:
{context}

Question: {question}

Answer:""")


def format_context(documents: list, max_tokens: int = 3000) -> str:
    """
    Format retrieved documents into a numbered context string.

    Args:
        documents: List of Document objects with .page_content and .metadata
        max_tokens: Approximate token limit for context (1 token ≈ 4 chars)

    Returns:
        Formatted string with numbered sources and metadata
    """
    context_parts = []
    total_chars = 0
    char_limit = max_tokens * 4  # Rough char-to-token conversion

    for i, doc in enumerate(documents):
        source = doc.metadata.get("source", "unknown")
        chunk_text = f"[Source {i+1}] (from: {source})\n{doc.page_content}\n"

        if total_chars + len(chunk_text) > char_limit:
            break

        context_parts.append(chunk_text)
        total_chars += len(chunk_text)

    return "\n---\n".join(context_parts)


# Full RAG chain:
rag_chain = (
    {"context": retriever | format_context, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)
answer = rag_chain.invoke("How do transformers work?")

# 7) Evaluation

RAG evaluation requires measuring BOTH retrieval AND generation quality.

Evaluation frameworks (2026):
  - RAGAS: Open-source, reference-free, pioneered context precision/recall
  - DeepEval: Pytest-style, self-explaining metrics, CI/CD integration
  - LangSmith: LangChain-native tracing and evaluation
  - Arize Phoenix: Open-source observability with embedding visualization
  - Braintrust: Production-to-evaluation feedback loop

Key metrics:
1. Retrieval metrics:
    - Context Precision: Are relevant items ranked higher?
    - Context Recall: Are all ground-truth items found in retrieved context?
    - Hit Rate: Does at least one relevant document appear in top-K?
    - NDCG: Normalized Discounted Cumulative Gain
    - MRR: Mean Reciprocal Rank

2. Generation metrics:
    - Faithfulness: Is the answer factually grounded in retrieved context?
    - Answer Relevancy: Does the answer address the original question?
    - Hallucination rate: Does the answer contain unsupported claims?

Evaluation approach:
  1. Component-level: Test retriever and generator separately
  2. End-to-end: Test the full pipeline on realistic queries
  3. Human-in-the-loop: Expert review for high-stakes domains

## 7.1 RAGAS

In [ ]:
import sys
import types

# ragas 0.4.x still imports ChatVertexAI from a module path that
# langchain-community 0.4 removed. Register a stub so the import resolves;
# we evaluate with OpenAI, so Vertex is never actually called.
for _name in ("langchain_community.chat_models.vertexai", "langchain_community.llms.vertexai"):
    _stub = types.ModuleType(_name)
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    _stub.VertexAI = type("VertexAI", (), {})
    sys.modules[_name] = _stub

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# RAGAS 0.4 takes a list of SingleTurnSample instead of a HuggingFace Dataset.
# Field names map as: question→user_input, answer→response,
# contexts→retrieved_contexts, ground_truths→reference (a single string).
samples = [
    SingleTurnSample(
        user_input="How do neural networks learn?",
        response="Neural networks learn hierarchical representations through backpropagation.",
        retrieved_contexts=[
            "Neural networks learn hierarchical representations of data."
        ],
        reference="Neural networks learn by adjusting weights through backpropagation.",
    ),
    SingleTurnSample(
        user_input="What is gradient descent?",
        response="Gradient descent follows the loss gradient to optimize parameters.",
        retrieved_contexts=[
            "Gradient descent optimizes model parameters by following the loss gradient."
        ],
        reference="Gradient descent minimizes loss by iteratively updating parameters.",
    ),
]

# In production, fill response/retrieved_contexts from your own pipeline:
# response=rag_chain.invoke(q)
# retrieved_contexts=[d.page_content for d in retriever.invoke(q)]

eval_dataset = EvaluationDataset(samples=samples)

# The judge model and embeddings RAGAS uses to score each metric
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

result = evaluate(
    dataset=eval_dataset,
    metrics=[
        # Retrieval: are relevant contexts ranked high / all found?
        LLMContextPrecisionWithReference(),
        LLMContextRecall(),
        # Generation: is the answer grounded, and does it address the question?
        Faithfulness(),
        ResponseRelevancy(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
print(result)
result.to_pandas()

{'llm_context_precision_with_reference': 0.5000, 'context_recall': 0.2500, 'faithfulness': 0.7500, 'answer_relevancy': 0.7375}


,user_input,retrieved_contexts,response,reference,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,How do neural networks learn?,[Neural networks learn hierarchical representa...,Neural networks learn hierarchical representat...,Neural networks learn by adjusting weights thr...,0.0,0.5,0.5,0.684116
1,What is gradient descent?,[Gradient descent optimizes model parameters b...,Gradient descent follows the loss gradient to ...,Gradient descent minimizes loss by iteratively...,1.0,0.0,1.0,0.790972


 ## 7.2 NDCG (Normalized Discounted Cumulative Gain)

 NDCG measures ranking quality considering graded relevance.

$$
DCG@K = Σ_{i=1}^{K} (2^{rel_i} - 1) / log2(i + 1)
$$

$$
NDCG@K = DCG@K / IDCG@K
$$

Where IDCG is DCG with ideal (sorted) ranking.

Intuition:
  - Relevant documents at higher ranks contribute more to the score
  - The log2 discount means rank 1 is ~6x more valuable than rank 10

In [ ]:
def compute_ndcg(
    predicted_items: list, ground_truth_items: list, top_k: int = 5
) -> float:
    """
    Compute NDCG@K for a single query.

    Args:
        predicted_items: Ranked list of retrieved item IDs
        ground_truth_items: Set of relevant item IDs
        top_k: Cutoff rank

    Returns:
        NDCG score between 0 and 1
    """
    # Binary relevance: 1 if item is in ground truth, 0 otherwise

    relevance = [
        1.0 if item in ground_truth_items else 0.0 
        for item in predicted_items[:top_k]
    ]

    # DCG: Discounted Cumulative Gain
    dcg = sum(
        (2**rel - 1) / math.log2(rank + 2)  # rank+2 because rank is 0-indexed
        for rank, rel in enumerate(relevance)
    )

    # Ideal DCG: Sort relevance scores descending
    ideal_relevance = sorted(relevance, reverse=True)
    idcg = sum(
        (2**rel - 1) / math.log2(rank + 2) for rank, rel in enumerate(ideal_relevance)
    )

    return dcg / idcg if idcg > 0 else 0.0


# Test NDCG
print("NDCG (perfect):  ", compute_ndcg([1, 2, 3], [1, 2], top_k=3))
print("NDCG (ok):       ", compute_ndcg([1, 3, 2], [2, 3], top_k=3))
print("NDCG (worst):    ", compute_ndcg([1, 2, 3], [3], top_k=3))

NDCG (perfect):   1.0
NDCG (ok):        0.6934264036172708
NDCG (worst):     0.5


## 7.3 Mean Reciprocal Rank (MRR)

MRR measures how quickly the first relevant result appears.

$$
MRR = (1/|Q|) * Σ_{i=1}^{|Q|} 1/rank_i
$$

Where rank_i is the rank of the first relevant document for query i.
MRR = 1.0 means the relevant doc is always rank 1.

In [33]:
def compute_mrr(result_lists: list[list[int]]) -> float:
    """
    Compute Mean Reciprocal Rank across multiple queries.

    Args:
        result_lists: List of binary relevance lists.
                      E.g., [[1,0,0], [0,1,0], [0,0,1]]
                      where 1 = relevant, 0 = not relevant

    Returns:
        MRR score between 0 and 1
    """
    reciprocal_ranks = []
    for results in result_lists:
        for rank, relevant in enumerate(results):
            if relevant:
                reciprocal_ranks.append(1.0 / (rank + 1))
                break
        else:
            reciprocal_ranks.append(0.0)

    return np.mean(reciprocal_ranks)


# Test MRR
print("MRR:", compute_mrr([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))  # (1 + 0.5 + 0.33)/3

MRR: 0.611111111111111


## 7.4 Hit Rate

In [34]:
def compute_hit_rate(
    predictions: list[list], ground_truths: list[list], top_k: int = 5
) -> float:
    """
    Compute Hit Rate@K: fraction of queries with at least one relevant
    document in the top-K results.

    Args:
        predictions: List of ranked item lists per query
        ground_truths: List of relevant item lists per query
        top_k: Number of top results to consider

    Returns:
        Hit rate between 0 and 1
    """
    hits = 0
    for pred, gt in zip(predictions, ground_truths):
        if any(item in gt for item in pred[:top_k]):
            hits += 1
    return hits / len(predictions)

## 7.5 DeepEval Integration

DeepEval: Pytest-style LLM evaluation with self-explaining metrics.

In [35]:
from deepeval import assert_test
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
)
from deepeval.test_case import LLMTestCase

# Create test case
test_case = LLMTestCase(
    input="What is gradient descent?",
    actual_output="Gradient descent minimizes loss by following the gradient.",
    expected_output="Gradient descent iteratively updates parameters to minimize loss.",
    retrieval_context=["Gradient descent optimizes by following the loss gradient."],
)

# Define metrics
faithfulness = FaithfulnessMetric(threshold=0.7)
relevancy = AnswerRelevancyMetric(threshold=0.7)


# Run as pytest
def test_rag_output():
    assert_test(test_case, [faithfulness, relevancy])


# Run in bulk
from deepeval import evaluate

evaluate([test_case], [faithfulness, relevancy])

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Faithfulness               │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Evaluation completed 🎉! (time taken: 5.25s | token cost: 0.009535000000000002 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions—the actual output appears fully consistent with the retrieval context. Nice job keeping it accurate and aligned.', strict_mode=False, flaky=False, evaluation_model='gpt-5.4', error=None, evaluation_cost=0.005215000000000001, input_tokens=1408, output_tokens=113, verbose_logs='Truths (limit=None):\n[\n    "Gradient descent optimizes by following the loss gradient."\n] \n \nClaims:\n[\n    "Gradient descent minimizes loss by following the gradient."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the response is fully relevant to the question about gradient descent, with no irrelevant statements—clear and on-topic th

# 8) FineTuning Embedding Model

Fine-tuning an embedding model on your domain data can improve retrieval accuracy by 10-30% for specialized domains (legal, medical, code).

Dataset formats:
1. Pairs + float label: (sentence_A, sentence_B, similarity_score)
    → Loss: CosineSimilarityLoss
2. Pairs (positive only): (query, relevant_document)
    → Loss: MultipleNegativesRankingLoss (most common)
3. Sentence + integer label: (sentence, class_id)
    → Loss: BatchHardTripletLoss
4. Triplets: (anchor, positive, negative)
    → Loss: TripletLoss

## 8.1 Embedding Adapter (Lightweight Fine-Tuning)

Instead of fine-tuning the entire embedding model, learn a linear transform
that adjusts the embedding space for your domain.

Approach:
  1. Generate synthetic query-document pairs using an LLM
  2. Get embeddings for all queries and documents
  3. Train a matrix W such that: improved_query = W @ original_query
  4. Optimize cosine similarity between improved queries and relevant docs

This is much cheaper than full fine-tuning (~minutes vs hours).

In [40]:
class EmbeddingAdapter(nn.Module):
    """
    Lightweight adapter that learns a linear transform on query embeddings.

    The adapter maps query embeddings into a space where they are
    closer to relevant document embeddings and farther from irrelevant ones.

    Architecture: Single linear layer (no bias, no activation)
    Why no activation: We want to preserve the geometric structure of the
    embedding space. A linear transform rotates/scales the space.
    """

    def __init__(self, embed_dim: int):
        super().__init__()
        # Learnable transformation matrix
        # Shape: (embed_dim, embed_dim)
        # Initialized near identity for stable training
        self.transform = nn.Parameter(
            torch.eye(embed_dim) + 0.01 * torch.randn(embed_dim, embed_dim)
        )

    def forward(self, query_embedding: torch.Tensor) -> torch.Tensor:
        """
        Apply learned transform to query embeddings.

        Input: (batch_num, embed_dim) — original query embeddings
        Output: (batch_num, embed_dim) — adapted query embeddings
        """
        # Matrix multiply: (batch_num, embed_dim) @ (embed_dim, embed_dim).T
        # Output: (batch_num, embed_dim)
        adapted = torch.matmul(query_embedding, self.transform.T)
        return F.normalize(adapted, p=2, dim=-1)


def train_adapter(
    query_embeddings: torch.Tensor,
    doc_embeddings: torch.Tensor,
    labels: torch.Tensor,
    embed_dim: int,
    epochs: int = 100,
    lr: float = 0.01,
) -> EmbeddingAdapter:
    """
    Train an embedding adapter on query-document pairs.

    Args:
        query_embeddings: (num_samples, embed_dim)
        doc_embeddings:   (num_samples, embed_dim)
        labels:           (num_samples,) — 1 for relevant, -1 for irrelevant
        embed_dim:        Embedding dimension
        epochs:           Training epochs
        lr:               Learning rate

    Returns:
        Trained EmbeddingAdapter
    """
    adapter = EmbeddingAdapter(embed_dim)
    optimizer = torch.optim.Adam(adapter.parameters(), lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Apply adapter to query embeddings
        # Input: (num_samples, embed_dim)
        # Output: (num_samples, embed_dim)
        adapted_queries = adapter(query_embeddings)

        # Compute cosine similarity
        # Input: (num_samples, embed_dim) pairs
        # Output: (num_samples,)
        similarities = F.cosine_similarity(adapted_queries, doc_embeddings, dim=1)

        # MSE loss: push similarity toward labels (1 or -1)
        loss = F.mse_loss(similarities, labels)

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    return adapter

# 9)  PRODUCTION DEPLOYMENT PATTERNS

In [41]:
class SemanticCache:
    """
    Cache RAG responses for semantically similar queries.

    Strategy:
      1. Embed the incoming query
      2. Check if any cached query has cosine similarity > threshold
      3. If yes: return cached response (cache hit)
      4. If no: run full RAG pipeline, cache result (cache miss)

    This dramatically reduces latency and cost for repeated/similar queries.
    """

    def __init__(self, embedding_model, similarity_threshold: float = 0.95):
        self.embedding_model = embedding_model
        self.threshold = similarity_threshold
        self.cache = {}  # query_embedding_key → response
        self.embeddings = []  # List of cached query embeddings
        self.keys = []  # Corresponding cache keys

    def _get_embedding(self, text: str) -> np.ndarray:
        return self.embedding_model.encode([text], normalize_embeddings=True)[0]

    def get(self, query: str) -> str | None:
        """Check cache for semantically similar query."""
        if not self.embeddings:
            return None

        query_emb = self._get_embedding(query)

        # Compute similarities with all cached queries
        # (1, embed_dim) @ (num_cached, embed_dim).T → (num_cached,)
        similarities = np.dot(np.array(self.embeddings), query_emb)

        max_idx = np.argmax(similarities)
        if similarities[max_idx] >= self.threshold:
            return self.cache[self.keys[max_idx]]

        return None

    def set(self, query: str, response: str):
        """Cache a query-response pair."""
        query_emb = self._get_embedding(query)
        key = hash(query)

        self.cache[key] = response
        self.embeddings.append(query_emb)
        self.keys.append(key)